# Baseline Model — mT5-base + LoRA Fine-Tuning

**Experiment 1:** Fine-tune mT5-base with LoRA on the full training set for 1 epoch.

**Setup:**
- Model: google/mt5-base
- Fine-tuning: LoRA (r=16, alpha=32)
- Epochs: 1
- Learning rate: 5e-4
- max_input_length: 128
- max_target_length: 256

In [1]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: Tesla T4


In [2]:
data_path = '/kaggle/input/datasets/oglorypaul/multilingual-health-qa/'

train = pd.read_csv(data_path + 'Train.csv')
val   = pd.read_csv(data_path + 'Val.csv')
test  = pd.read_csv(data_path + 'Test.csv')

print(f'Train: {train.shape}')
print(f'Val  : {val.shape}')
print(f'Test : {test.shape}')

Train: (29815, 4)
Val  : (6686, 4)
Test : (2618, 3)


In [3]:
MODEL_NAME = 'google/mt5-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Tokenizer loaded! Vocab size: {tokenizer.vocab_size:,}')

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

Tokenizer loaded! Vocab size: 250,100


In [4]:
def preprocess(examples):
    inputs = tokenizer(
        examples['input'],
        max_length=128,
        truncation=True,
        padding='max_length'
    )
    targets = tokenizer(
        examples['output'],
        max_length=256,
        truncation=True,
        padding='max_length'
    )
    inputs['labels'] = targets['input_ids']
    return inputs

train_dataset = Dataset.from_pandas(train[['input', 'output']])
val_dataset   = Dataset.from_pandas(val[['input', 'output']])

train_tokenized = train_dataset.map(preprocess, batched=True)
val_tokenized   = val_dataset.map(preprocess, batched=True)

print(f'Train tokenized: {len(train_tokenized)}')
print(f'Val tokenized  : {len(val_tokenized)}')

Map:   0%|          | 0/29815 [00:00<?, ? examples/s]

Map:   0%|          | 0/6686 [00:00<?, ? examples/s]

Train tokenized: 29815
Val tokenized  : 6686


In [5]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=['q', 'v']
)

model = get_peft_model(model, lora_config)
model = model.to(device)
model.print_trainable_parameters()

pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

trainable params: 1,769,472 || all params: 968,342,784 || trainable%: 0.1827


## Experiment 1 — LoRA Fine-Tuning, 1 Epoch
**Hypothesis:** Fine-tuning mT5-base with LoRA for 1 epoch will produce meaningful answers and a measurable ROUGE score.
**Changes from baseline:**
- Added LoRA (r=16, alpha=32, dropout=0.1)
- fp16=False (full precision for stability)
- 1 epoch, lr=5e-4
- max_input_length=128, max_target_length=256

**Result:** Training Loss=7.60, Val Loss=3.21, Zindi Score=0.182709
**Insight:** Model learned to generate meaningful African language text. Repetition observed in outputs suggests need for longer training or lower learning rate.

In [6]:
training_args = Seq2SeqTrainingArguments(
    output_dir='/kaggle/working/exp1_lora',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    warmup_steps=500,
    learning_rate=5e-4,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    predict_with_generate=True,
    fp16=False,
    report_to='none'
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

print('Trainer ready!')

Trainer ready!


In [ ]:
trainer.train()
print('Training complete!')

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss


In [ ]:
trainer.save_model('/kaggle/working/exp1_final')
tokenizer.save_pretrained('/kaggle/working/exp1_final')
print('Saved!')

In [ ]:
from tqdm import tqdm

model.eval()

def generate_batch(questions, max_new_tokens=128):
    inputs = tokenizer(
        questions,
        return_tensors='pt',
        max_length=128,
        truncation=True,
        padding=True
    ).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            no_repeat_ngram_size=3
        )
    
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

predictions = []
batch_size = 16

for i in tqdm(range(0, len(test), batch_size)):
    batch = test['input'].iloc[i:i+batch_size].tolist()
    preds = generate_batch(batch)
    predictions.extend(preds)

print(f'Generated {len(predictions)} predictions')
print(f'Sample: {predictions[0][:150]}')

In [ ]:
sample_sub = pd.read_csv(data_path + 'SampleSubmission.csv')

submission = sample_sub.copy()
submission['TargetRLF1'] = predictions
submission['TargetR1F1'] = predictions
submission['TargetLLM']  = predictions

submission.to_csv('/kaggle/working/submission.csv', index=False)
print('Submission saved!')
print(submission.head(3))

## Experiment 1 — Summary
**Setup:** mT5-base + LoRA (r=16), 1 epoch, lr=5e-4, fp16=False
**Zindi Score: 0.182709**
**What worked:** Model successfully generated meaningful text in African languages — a significant improvement over zero-shot gibberish.
**What didn't:** Some repetition in outputs and training loss (7.60) suggests the model needs more epochs or a lower learning rate.
**Next:** Increase to 3 epochs with lower learning rate to improve score.

## Experiment 2 — 3 Epochs, Lower Learning Rate
**Hypothesis:** Training for 3 epochs with a lower learning rate (1e-4) will reduce repetition and improve ROUGE score.
**Changes from Experiment 1:**
- num_train_epochs: 1 → 3
- learning_rate: 5e-4 → 1e-4
**Expected outcome:** Higher Zindi score, less repetition in outputs

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()
print('Memory cleared!')

In [ ]:
model_exp2 = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=['q', 'v']
)

model_exp2 = get_peft_model(model_exp2, lora_config)
model_exp2 = model_exp2.to(device)
model_exp2.print_trainable_parameters()

In [ ]:
training_args_exp2 = Seq2SeqTrainingArguments(
    output_dir='/kaggle/working/exp2_lora_3epochs',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    warmup_steps=500,
    learning_rate=1e-4,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    predict_with_generate=True,
    fp16=False,
    report_to='none'
)

data_collator_exp2 = DataCollatorForSeq2Seq(tokenizer, model=model_exp2, padding=True)

trainer_exp2 = Seq2SeqTrainer(
    model=model_exp2,
    args=training_args_exp2,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator_exp2,
)

trainer_exp2.train()
print('Experiment 2 training complete!')

## Experiment 2 — Results & Summary
**Training output observed:**

| Epoch | Training Loss | Validation Loss |
|---|---|---|
| 1 | 9.144973 | 3.727624 |
| 2 | 19.081573 | 8.875292 |
| 3 | 11.100088 | 5.972592 |

**Zindi Score: 0.095423** — roughly half of Experiment 1's score (0.182709)

**Insight:** Loss rises rather than falls after epoch 1, a clear sign of training instability rather than productive learning. Combined with the lower learning rate (1e-4) and extended training (3 epochs), this configuration moved the model away from a good solution rather than toward one. This confirmed that Experiment 1's settings (1 epoch, lr=5e-4) should remain the default going forward, and additional epochs require a specific justification before being tested again.